In [1]:
tabla='qtiop10'
col_tabla = "infopefec"
dat= "dat_ceqx002_essi"
col_essi='fec_ope'
col_dat='fec_ope'
essi='essi_dat_cqx002_etl'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [6]:
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)    
now = datetime.now()
query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

c1= text(query)
connection2.execute(c1)

In [7]:

fecha= '2023-05-01'

In [8]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)

cqxrespat=pd.read_sql_query(f"SELECT id_respat,cod_res FROM dim_cqxrespat", con=connection2)
cqxrespat['cod_res']=cqxrespat['cod_res'].str.strip()

cqxtipher=pd.read_sql_query(f"SELECT id_tipher,cod_the FROM dim_cqxtipher", con=connection2)
cqxtipher['cod_the']=cqxtipher['cod_the'].str.strip()

cqxdesegrsop=pd.read_sql_query(f"SELECT id_desegr,des_cod FROM dim_cqxdesegrsop", con=connection2)
cqxdesegrsop['des_cod']=cqxdesegrsop['des_cod'].str.strip()

numdoc = pd.read_sql_query(f"SELECT id_person,num_doc FROM dim_personal", con=connection2)
numdoc["num_doc"]=numdoc["num_doc"].str.strip()
numdoc["num_doc"]=numdoc["num_doc"].str.replace(' ', '', regex=True)
numdoc=numdoc.drop_duplicates(subset="num_doc")

In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [10]:
base1.shape

(48664, 33)

In [11]:
#Eliminamos las columnas que no se usarán aquí: las descripciones y otras 4 más, previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_des', 'des_her', 'des_red', 'des_res']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [12]:
base1.shape

(48664, 28)

In [13]:
inicioProceso=time.time()

In [14]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [15]:
base1.shape

(48664, 28)

In [16]:
control_a.append(base1.shape[0])

In [17]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(48664, 29)

In [18]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [19]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(48664, 29)

In [20]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [21]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(48664, 29)

In [22]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [23]:
numdoc=numdoc.rename(columns={"num_doc": "usu_mod","id_person": "id_usumod"})
base1['usu_mod']=base1['usu_mod'].str.strip()
base1["usu_mod"]=base1["usu_mod"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='usu_mod')
base1=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1.shape

(48664, 29)

In [24]:
merge.shape #Se pierden datos

(21572, 29)

In [25]:
log_falla('id_usumod', 'usu_mod', True)
log_control('dim_personal')
base1=base1.drop('usu_mod',axis=1)

In [26]:
numdoc=numdoc.rename(columns={"usu_mod": "usu_cre","id_usumod": "id_usucred"})
base1['usu_cre']=base1['usu_cre'].str.strip()
base1["usu_cre"]=base1["usu_cre"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='usu_cre')
base1=pd.merge(base1,numdoc,how='left',on='usu_cre') 
base1.shape

(48664, 29)

In [27]:
merge.shape #Se pierden muy pocos datos

(48645, 29)

In [28]:
log_falla('id_usucred', 'usu_cre', True)
log_control('dim_personal')
base1=base1.drop('usu_cre',axis=1)

In [29]:
cqxrespat=cqxrespat.rename(columns={"cod_res": "res_pat"})
base1['res_pat']=base1['res_pat'].str.strip()
base1["res_pat"]=base1["res_pat"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxrespat,how='left',on='res_pat')
base1=pd.merge(base1,cqxrespat,how='inner',on='res_pat')
base1.shape

(48664, 29)

In [30]:
merge.shape

(48664, 29)

In [31]:
log_falla('id_respat', 'res_pat', True)
log_control('dim_cqxrespat')
base1=base1.drop('res_pat',axis=1)

In [32]:
cqxtipher=cqxtipher.rename(columns={"cod_the": "her_ope","id_tipher": "id_herope"})
base1['her_ope']=base1['her_ope'].str.strip()
base1["her_ope"]=base1["her_ope"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxtipher,how='left',on='her_ope')
base1=pd.merge(base1,cqxtipher,how='inner',on='her_ope')
base1.shape

(48664, 29)

In [33]:
merge.shape

(48664, 29)

In [34]:

log_falla('id_herope', 'her_ope', True)
log_control('dim_cqxtipher')
base1=base1.drop('her_ope',axis=1)

In [35]:
cqxdesegrsop=cqxdesegrsop.rename(columns={"des_cod": "cod_des","id_desegr": "id_dessop"})
base1['cod_des']=base1['cod_des'].str.strip()
base1["cod_des"]=base1["cod_des"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxdesegrsop,how='left',on='cod_des')
base1=pd.merge(base1,cqxdesegrsop,how='inner',on='cod_des')
base1.shape

(48664, 29)

In [36]:
merge.shape

(48664, 29)

In [37]:
log_falla('id_dessop', 'cod_des', True)
base1=base1.drop('cod_des',axis=1)
dim.append('dim_cqxdesegrsop')
control_d.append(base1.shape[0])

In [38]:
base1['fec_ope_temp'] = base1['fec_ope'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"dt_fecha":"fec_ope_temp"})
tiempo["fec_ope_temp"] = tiempo['fec_ope_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_ope_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_ope_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_ope","dt_ano":"ano_ope","dt_mes":"mes_ope",
                            "dt_dia":"dia_ope","dt_dia_sem":"dia_sem_ope","dt_sem":"sem_ope"})
base1=base1.drop("fec_ope_temp",axis=1)
base1.shape

(48664, 34)

In [39]:
tiempo

,id_tiempo,fec_ope_temp,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano
0,20130101,2013-01-01,1.0,1.0,2.0,1.0,2013.0
1,20130102,2013-01-02,1.0,2.0,3.0,1.0,2013.0
2,20130103,2013-01-03,1.0,3.0,4.0,1.0,2013.0
3,20130104,2013-01-04,1.0,4.0,5.0,1.0,2013.0
4,20130105,2013-01-05,1.0,5.0,6.0,1.0,2013.0
...,...,...,...,...,...,...,...
5474,20261227,2026-12-27,12.0,27.0,7.0,2.0,2026.0
5475,20261228,2026-12-28,12.0,28.0,1.0,2.0,2026.0
5476,20261229,2026-12-29,12.0,29.0,2.0,2.0,2026.0
5477,20261230,2026-12-30,12.0,30.0,3.0,2.0,2026.0


In [40]:
base1['fec_cre_temp'] = base1['fec_cre'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"fec_ope_temp":"fec_cre_temp"})
tiempo["fec_cre_temp"] = tiempo['fec_cre_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_cre_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_cre_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_cre","dt_ano":"ano_cre","dt_mes":"mes_cre",
                            "dt_dia":"dia_cre","dt_dia_sem":"dia_sem_cre","dt_sem":"sem_cre"})
base1=base1.drop("fec_cre_temp",axis=1)
base1.shape

(48664, 40)

In [41]:
base1['fec_mod_temp'] = base1['fec_mod'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"fec_cre_temp":"fec_mod_temp"})
tiempo["fec_mod_temp"] = tiempo['fec_mod_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_mod_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_mod_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_mod","dt_ano":"ano_mod","dt_mes":"mes_mod",
                            "dt_dia":"dia_mod","dt_dia_sem":"dia_sem_mod","dt_sem":"sem_mod"})
base1=base1.drop("fec_mod_temp",axis=1)
base1.shape

(48664, 46)

In [42]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48664 entries, 0 to 48663
Data columns (total 46 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   num_sol      48664 non-null  float64            
 1   num_sec      48664 non-null  float64            
 2   fec_ope      48664 non-null  object             
 3   hor_ing_sop  48664 non-null  object             
 4   tie_uso_sop  48664 non-null  object             
 5   tie_ane      48664 non-null  object             
 6   tie_ope      48664 non-null  object             
 7   fec_cre      48664 non-null  datetime64[ns, UTC]
 8   fec_mod      22344 non-null  datetime64[ns, UTC]
 9   act_med      48664 non-null  float64            
 10  inf_peb      48661 non-null  float64            
 11  inf_rog      48661 non-null  float64            
 12  inf_per      48661 non-null  float64            
 13  inf_ces      48661 non-null  object             
 14  inf_tra      48661 non

In [43]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [44]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [45]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [46]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [47]:
base

,hor_ing_sop,id_usucred,fec_mod,id_herope,pac_sec,id_time_mod,dia_ope,inf_exp,tie_ope,dia_sem_mod,...,mes_ope,mes_cre,fec_ope,num_sec,id_red,id_oricas,inf_tra,id_time_ope,inf_are,dia_cre
0,0001-01-01 07:20:00-04:56:16,6678630.0,NaT,1,8945335.0,NaN,1.0,0.0,0001-01-01 01:30:00-04:56:16,NaN,...,6.0,6.0,2023-06-01,1.0,11,1,0.0,20230601,0.0,1.0
1,0001-01-01 08:30:00-04:56:16,6658647.0,NaT,1,27849171.0,NaN,12.0,0.0,0001-01-01 01:20:00-04:56:16,NaN,...,5.0,5.0,2023-05-12,1.0,11,1,0.0,20230512,0.0,12.0
2,0001-01-01 07:50:00-04:56:16,6751036.0,NaT,1,22527083.0,NaN,8.0,0.0,0001-01-01 01:00:00-04:56:16,NaN,...,5.0,5.0,2023-05-08,1.0,11,1,0.0,20230508,0.0,8.0
3,0001-01-01 14:00:00-04:56:16,6745928.0,NaT,1,19633831.0,NaN,9.0,0.0,0001-01-01 01:40:00-04:56:16,NaN,...,5.0,5.0,2023-05-09,1.0,11,1,0.0,20230509,0.0,9.0
4,0001-01-01 14:00:00-04:56:16,6745928.0,NaT,1,10195950.0,NaN,8.0,0.0,0001-01-01 02:00:00-04:56:16,NaN,...,5.0,5.0,2023-05-08,1.0,11,1,0.0,20230508,0.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48659,0001-01-01 08:30:00-04:56:16,6751056.0,NaT,1,10123775.0,NaN,16.0,0.0,0001-01-01 05:00:00-04:56:16,NaN,...,6.0,6.0,2023-06-16,1.0,11,1,0.0,20230616,0.0,16.0
48660,0001-01-01 10:45:00-04:56:16,6711964.0,2023-06-14 19:50:04+00:00,1,17314101.0,20230614.0,14.0,0.0,0001-01-01 01:20:00-04:56:16,3.0,...,6.0,6.0,2023-06-14,1.0,3,1,0.0,20230614,0.0,14.0
48661,0001-01-01 21:45:00-04:56:16,6728751.0,2023-05-16 07:25:12+00:00,1,25371735.0,20230516.0,15.0,0.0,0001-01-01 00:00:00-04:56:16,2.0,...,5.0,5.0,2023-05-15,1.0,2,1,0.0,20230515,0.0,16.0
48662,0001-01-01 09:10:00-04:56:16,6723464.0,NaT,1,10588790.0,NaN,6.0,0.0,0001-01-01 00:35:00-04:56:16,NaN,...,5.0,5.0,2023-05-06,1.0,7,1,0.0,20230506,0.0,6.0


In [48]:
base.columns

Index(['hor_ing_sop', 'id_usucred', 'fec_mod', 'id_herope', 'pac_sec',
       'id_time_mod', 'dia_ope', 'inf_exp', 'tie_ope', 'dia_sem_mod',
       'id_time_cre', 'num_sol', 'id_usumod', 'sol_act', 'dia_sem_ope',
       'act_med', 'sem_ope', 'sem_mod', 'inf_peb', 'fec_cre', 'id_respat',
       'ano_mod', 'sem_cre', 'dia_sem_cre', 'ano_ope', 'inf_rog', 'ano_cre',
       'id_dessop', 'inf_per', 'dia_mod', 'mes_mod', 'inf_ces', 'sol_fec',
       'tie_uso_sop', 'tie_ane', 'id_cas', 'mes_ope', 'mes_cre', 'fec_ope',
       'num_sec', 'id_red', 'id_oricas', 'inf_tra', 'id_time_ope', 'inf_are',
       'dia_cre'],
      dtype='object')

In [49]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

4664

In [50]:
fecha_fin = "2024-12-31"

In [51]:
proceso = "DAT"
cod_proceso = 13

# proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
# proceso = proceso.iloc[0, 0]
# cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
# cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [52]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_oricas,2023-05-01,2024-12-31,48664,48664,0,0,id_oricas
1,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_cas,2023-05-01,2024-12-31,48664,48664,0,0,id_cas
2,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_red,2023-05-01,2024-12-31,48664,48664,0,0,id_red
3,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_personal,2023-05-01,2024-12-31,48664,48664,0,0,id_usumod
4,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_personal,2023-05-01,2024-12-31,48664,48664,0,0,id_usucred
5,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_cqxrespat,2023-05-01,2024-12-31,48664,48664,0,0,id_respat
6,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_cqxtipher,2023-05-01,2024-12-31,48664,48664,0,0,id_herope
7,13,DAT,dat_ceqx002_essi,2023-06-23,16:27,16:27,dim_cqxdesegrsop,2023-05-01,2024-12-31,48664,48664,0,0,id_dessop


In [53]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

8

In [54]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.3 completado satisfactoriamente en 19.685 segundos


In [55]:
connection1.close()
connection2.close()
connection3.close()
connection4.close()

In [56]:
engine1.dispose()
engine2.dispose()
engine3.dispose()
engine4.dispose()